# Cas9 DMS Activity Prediction
## Step 3 — Baseline Models

### Objective

Establish simple predictive baselines before introducing pretrained protein
sequence representations.

The main target is `DMS_score`, which is a continuous activity-related
phenotype from the Cas9 deep mutational scanning assay.

We evaluate every baseline under two protocols:

1. Position-held-out split
   - Test positions are completely unseen during training.
   - Primary evaluation.

2. Random mutation split
   - Different mutations from the same position may occur in different
     splits.
   - Secondary interpolation evaluation.

The baseline experiments establish how much predictive information is present
in simple mutation-level features before using pretrained protein models.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from scipy.stats import pearsonr, spearmanr

SEED = 42

In [2]:
def find_project_root(start_path):
    start_path = Path(start_path).resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path

    raise FileNotFoundError(
        "Could not find Mandrake-Bio project root."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Project root:", PROJECT_ROOT)
print("Results directory:", RESULTS_DIR)

Project root: C:\Users\subha\OneDrive\Desktop\Mandrake_Ai_Bio_research\Mandrake_Cas9_Assignment_Pack\Mandrake-Bio
Results directory: C:\Users\subha\OneDrive\Desktop\Mandrake_Ai_Bio_research\Mandrake_Cas9_Assignment_Pack\Mandrake-Bio\results


In [4]:
PROCESSED_DATA_PATH = (
    RESULTS_DIR / "processed_cas9_dms.csv"
)

df = pd.read_csv(PROCESSED_DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (8117, 7)


,mutant,mutated_sequence,DMS_score,DMS_score_bin,wt_residue,position,mutant_residue
0,A1023D,MDKKYSIGLDIGTNSVGWAVITDEYKVPSKKFKVLGNTDRHSIKKN...,-0.620198,0,A,1023,D
1,A1023G,MDKKYSIGLDIGTNSVGWAVITDEYKVPSKKFKVLGNTDRHSIKKN...,-0.424130,0,A,1023,G
2,A1023P,MDKKYSIGLDIGTNSVGWAVITDEYKVPSKKFKVLGNTDRHSIKKN...,-0.626792,0,A,1023,P
3,A1023S,MDKKYSIGLDIGTNSVGWAVITDEYKVPSKKFKVLGNTDRHSIKKN...,-0.490423,0,A,1023,S
4,A1023T,MDKKYSIGLDIGTNSVGWAVITDEYKVPSKKFKVLGNTDRHSIKKN...,-0.543741,0,A,1023,T
